# Phase 4 — TensorFlow LSTM Forecasting

This notebook trains a one-step-ahead multivariate LSTM for daily demand and evaluates it recursively for 7, 30 and 90 days.

**Data split:** train through 2023, validation 2024, final test 2025.

**Primary target:** Quantity / demand.

**Known inputs:** calendar variables, promotion and holiday flags.

**Historical inputs:** lagged demand represented by the input sequence.

> Run this notebook in Google Colab, where TensorFlow is normally available. The current execution environment does not have TensorFlow installed, so the training cell is intentionally provided as a ready-to-run experiment rather than claiming an unexecuted LSTM result.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from sklearn.preprocessing import StandardScaler

print('TensorFlow:', tf.__version__)
print('Num GPUs:', len(tf.config.list_physical_devices('GPU')))


In [ ]:
DATA_CANDIDATES = [
    Path('/mnt/data/processed_daily_forecasting_features.csv'),
    Path('/content/processed_daily_forecasting_features.csv'),
    Path('/mnt/data/smart_sales_forecasting_dataset/processed_daily_forecasting_features.csv'),
]
data_path = next((p for p in DATA_CANDIDATES if p.exists()), None)
if data_path is None:
    raise FileNotFoundError('Upload processed_daily_forecasting_features.csv to Colab first.')

df = pd.read_csv(data_path, parse_dates=['Date'])
df = df.sort_values('Date').reset_index(drop=True)

print('Dataset:', df.shape)
display(df.head())

## 1. Prepare an LSTM modeling table

We deliberately use only information that can be available at forecast time. No future Quantity or Sales_Amount is included in the input.

In [ ]:
feature_cols = [
    'Quantity',
    'Promotions',
    'Holiday_Flag',
    'day_of_week',
    'day_of_month',
    'week_of_year',
    'month',
    'quarter',
    'is_weekend',
    'days_since_start',
]

model_df = df[['Date'] + feature_cols].copy()
model_df = model_df.dropna().reset_index(drop=True)

train_df = model_df[model_df.Date < '2024-01-01'].copy()
val_df = model_df[(model_df.Date >= '2024-01-01') & (model_df.Date < '2025-01-01')].copy()
test_df = model_df[model_df.Date >= '2025-01-01'].copy()

print('Train:', train_df.Date.min().date(), 'to', train_df.Date.max().date(), len(train_df))
print('Validation:', val_df.Date.min().date(), 'to', val_df.Date.max().date(), len(val_df))
print('Test:', test_df.Date.min().date(), 'to', test_df.Date.max().date(), len(test_df))

## 2. Scale without leakage

The scaler is fitted only on the training period. Validation and test data are transformed using the training scaler.

In [ ]:
feature_scaler = StandardScaler()
feature_scaler.fit(train_df[feature_cols])

train_scaled = feature_scaler.transform(train_df[feature_cols])
val_scaled = feature_scaler.transform(val_df[feature_cols])
test_scaled = feature_scaler.transform(test_df[feature_cols])

target_idx = feature_cols.index('Quantity')
quantity_mean = feature_scaler.mean_[target_idx]
quantity_scale = feature_scaler.scale_[target_idx]

def inverse_quantity(x):
    return np.asarray(x) * quantity_scale + quantity_mean

## 3. Build 28-day sequences

Each sample uses the previous 28 daily observations to predict the next day's demand.

In [ ]:
LOOKBACK = 28

def make_sequences(values, lookback=28):
    X, y = [], []
    for i in range(lookback, len(values)):
        X.append(values[i-lookback:i])
        y.append(values[i, target_idx])
    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32)

X_train, y_train = make_sequences(train_scaled, LOOKBACK)
X_val, y_val = make_sequences(val_scaled, LOOKBACK)

print('X_train:', X_train.shape, 'y_train:', y_train.shape)
print('X_val:', X_val.shape, 'y_val:', y_val.shape)

## 4. LSTM architecture

Architecture: 2 LSTM layers → dropout → dense output. Early stopping restores the best validation weights.

In [ ]:
tf.keras.utils.set_random_seed(42)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(LOOKBACK, len(feature_cols))),
    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.Dropout(0.20),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dropout(0.20),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',
    metrics=[tf.keras.metrics.MeanAbsoluteError()]
)

model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=7, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=40,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(history.history['loss'], label='Train Loss')
ax.plot(history.history['val_loss'], label='Validation Loss')
ax.set_title('LSTM Training History')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Recursive multi-day forecasting

The model predicts one day at a time. For future days, the previous prediction is fed back as Quantity, while known calendar/promotion/holiday values are taken from the dataset.

In [ ]:
known = model_df.set_index('Date')

def recursive_lstm_forecast(model, history_df, future_dates, lookback=28):
    hist = history_df[['Date'] + feature_cols].copy().sort_values('Date').reset_index(drop=True)
    rows = []

    for date in pd.DatetimeIndex(future_dates):
        if date not in known.index:
            raise KeyError(f'No known future calendar/exogenous values for {date}')

        last = hist.tail(lookback).copy()
        scaled_window = feature_scaler.transform(last[feature_cols])
        x = scaled_window.reshape(1, lookback, len(feature_cols)).astype(np.float32)
        pred_scaled = float(model.predict(x, verbose=0)[0, 0])
        pred = float(max(0.0, inverse_quantity(pred_scaled)))

        source = known.loc[date]
        new_row = {
            'Date': date,
            'Quantity': pred,
            'Promotions': source['Promotions'],
            'Holiday_Flag': source['Holiday_Flag'],
            'day_of_week': date.dayofweek,
            'day_of_month': date.day,
            'week_of_year': int(date.isocalendar().week),
            'month': date.month,
            'quarter': date.quarter,
            'is_weekend': int(date.dayofweek >= 5),
            'days_since_start': (date - model_df.Date.min()).days,
        }
        rows.append(new_row)
        hist = pd.concat([hist, pd.DataFrame([new_row])], ignore_index=True)

    return np.array([r['Quantity'] for r in rows])

def metric_table(actual, pred):
    actual, pred = np.asarray(actual, float), np.asarray(pred, float)
    err = actual - pred
    mae = np.mean(np.abs(err))
    rmse = np.sqrt(np.mean(err**2))
    nz = actual != 0
    mape = np.mean(np.abs(err[nz] / actual[nz])) * 100
    wape = np.sum(np.abs(err)) / np.sum(np.abs(actual)) * 100
    return mae, rmse, mape, wape

In [ ]:
test_history = model_df[model_df.Date < '2025-01-01'].copy()
rows = []

for horizon in [7, 30, 90]:
    future = test_df.head(horizon)
    pred = recursive_lstm_forecast(model, test_history, future.Date)
    actual = future.Quantity.to_numpy()
    mae, rmse, mape, wape = metric_table(actual, pred)
    rows.append({
        'Model': 'TensorFlow LSTM',
        'Target': 'Quantity',
        'Horizon_Days': horizon,
        'MAE': mae,
        'RMSE': rmse,
        'MAPE_%': mape,
        'WAPE_%': wape,
    })

lstm_results = pd.DataFrame(rows)
display(lstm_results.round(3))

## 6. Compare LSTM against Phase 3

Load the saved Phase 3 results and compare WAPE. The LSTM is not automatically the winner; the production model should be chosen by out-of-sample performance.

In [ ]:
phase3_candidates = [
    Path('/mnt/data/phase_3_ml_results.csv'),
    Path('/content/phase_3_ml_results.csv'),
]
phase3_path = next((p for p in phase3_candidates if p.exists()), None)

if phase3_path:
    phase3 = pd.read_csv(phase3_path)
    phase3_test = phase3[phase3.Evaluation == 'Test_2025'].copy()
    comparison = pd.concat([
        phase3_test[['Model','Horizon_Days','MAE','RMSE','MAPE_%','WAPE_%']],
        lstm_results[['Model','Horizon_Days','MAE','RMSE','MAPE_%','WAPE_%']]
    ], ignore_index=True)
    display(comparison.sort_values(['Horizon_Days','WAPE_%']).round(3))
else:
    print('Upload phase_3_ml_results.csv to compare automatically.')

## 7. Save the LSTM model and results


In [ ]:
model.save('sales_demand_lstm.keras')
lstm_results.to_csv('phase_4_lstm_results.csv', index=False)
print('Saved sales_demand_lstm.keras and phase_4_lstm_results.csv')

# Phase 4 decision rule

For each horizon, compare LSTM WAPE with the strongest Phase 3 model.

- If LSTM materially improves the error: consider LSTM the candidate for that horizon.
- If LSTM is worse: keep the stronger classical ML model.
- Do not select a model because it is more complex.

Next: integrate the selected model into a FastAPI forecasting service.